# Phenotypic Selection Tutorial - Clonal Breeding

This notebook replicates the AlphaSimR clonal breeding phenotypic selection tutorial using AlphaSimPy.
It demonstrates phenotypic selection in a clonal tea breeding program with multiple evaluation stages.

**Authors**: Translated from AlphaSimR tutorial by Nelson Lubanga, Gregor Gorjanc, Jon Bancic, Philip Greenspoon, Chris Gaynor  
**Date**: 2024  
**Package**: AlphaSimPy

## Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from AlphaSimPy import (
    run_macs, SimParam, new_pop, rand_cross, set_pheno, select_ind,
    mean_g, var_g
)

print("AlphaSimPy Clonal Breeding - Phenotypic Selection Tutorial")
print("All libraries imported successfully!")


## Global Parameters

Set up the simulation parameters for the clonal breeding program.

In [ ]:
# Number of simulation replications and breeding cycles
n_reps = 1  # Number of simulation replicates
n_burnin = 40  # Number of years in burnin phase
n_future = 40  # Number of years in future phase
start_records = 35  # Year when training and pedigree record collecting begins
n_cycles = n_burnin + n_future

# Genome simulation
n_chr = 15  # Number of chromosomes
n_qtl = 160  # Number of QTL per chromosome: 15 chr x 160 QTL = 2400 QTLs
n_snp = 600  # Simulate SNP chip with 9000 markers
gen_len = 1  # Genetic length
phy_len = 1e8  # Physical length
mut_rate = 2.5e-8  # Mutation rate

# Initial parents mean and variance
init_mean_g = 2500  # Phenotypic mean
init_var_g = 150000  # Genetic variance
init_var_ge = 150000  # Genotype-by-year interaction variance
var_e = 2800000  # Single variance

# Breeding program details
n_parents = 20  # Number of parents (and founders)
n_crosses = 100  # Number of crosses
n_progeny = 20  # Number of progenies per cross
n_clones_act = 500  # Number of individuals selected at ACT stage
n_clones_ect = 40  # Number of individuals selected at ECT stage

# Effective replication of yield trials
rep_hpt = 1  # h2 = 0.05
rep_act = 15  # h2 = 0.45
rep_ect = 50  # h2 = 0.65

scenario_name = "ClonalPheno"

print(f"Simulation Parameters:")
print(f"  Replicates: {n_reps}")
print(f"  Burn-in years: {n_burnin}")
print(f"  Future years: {n_future}")
print(f"  Total cycles: {n_cycles}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per chromosome: {n_qtl}")
print(f"  SNP per chromosome: {n_snp}")
print(f"  Parents: {n_parents}")
print(f"  Crosses per year: {n_crosses}")
print(f"  Progeny per cross: {n_progeny}")

## Create Founders

Generate the initial founder population with haplotypes and set up simulation parameters.

In [ ]:
print("Creating founders...")

# Create founder population
founder_pop = run_macs(
    n_ind=n_parents,
    n_chr=n_chr,
    seg_sites=n_qtl + n_snp,
)

print(f"✓ Created founder population: {founder_pop.n_ind} individuals")

# Set simulation parameters
SP = SimParam(founder_pop)

# Restrict segregating sites (separate QTL and SNP)
SP.restrSegSites(minQtlPerChr=n_qtl, minSnpPerChr=n_snp)

# Add SNP chip
if n_snp > 0:
    SP.addSnpChip(n_snp)
    print(f"✓ Added SNP chip: {SP.n_snp_chips} SNP chips")

# Add traits: trait represents yield
# Using addTraitADG for additive, dominance, and GxE effects
SP.addTraitADG(
    nQtlPerChr=n_qtl,
    mean=init_mean_g,
    var=init_var_g,
    var_gxE=init_var_ge
)
print(f"✓ Added TraitADG: {SP.n_traits} traits")

# Collect pedigree
SP.setTrackPed(True)
print("✓ Enabled pedigree tracking")

# Create founder parents
Parents = new_pop(founder_pop, sim_param=SP)
print(f"✓ Created founder parents: {Parents.n_ind} individuals, {Parents.n_traits} traits")

# Set a phenotype to founder parents
Parents = set_pheno(Parents, var_e=var_e, reps=rep_ect, sim_param=SP)

print(f"\nFounder population summary:")
print(f"  Mean genetic value: {mean_g(Parents)[0]:.3f}")
print(f"  Genetic variance: {var_g(Parents)[0]:.3f}")


## Fill Breeding Pipeline

Set up the initial breeding pipeline with 16 stages representing different evaluation years.
The pipeline includes:
- Stage 1: Crossing block (F1)
- Stages 2-4: Seedling evaluation (HPT1-3)
- Stages 5-9: Advanced clonal trials (ACT1-5)
- Stages 10-15: Elite clonal trials (ECT1-6)

In [ ]:
print("Filling breeding pipeline...")

# Set initial yield trials with unique individuals
# Sample year effects
P = np.random.uniform(size=16)

# Breeding program
for cohort in range(1, 17):
    print(f"  FillPipeline stage: {cohort} of 16")
    
    # Stage 1: Crossing block
    F1 = rand_cross(Parents, n_crosses=n_crosses, n_progeny=n_progeny, sim_param=SP)
    
    if cohort < 16:
        # Stage 2: Germinate the seedlings in the nursery
        Seedlings = set_pheno(F1, var_e=var_e, reps=rep_hpt, sim_param=SP)
    
    if cohort < 15:
        # Stage 3: Plant in the seedlings in the field as HPT and record yields
        HPT1 = Seedlings
    
    if cohort < 14:
        # Stage 4: Record the HPT yields
        HPT2 = HPT1
    
    if cohort < 13:
        # Stage 5: Record the HPT yields
        HPT3 = set_pheno(HPT2, var_e=var_e, reps=rep_hpt, sim_param=SP)
    
    if cohort < 12:
        # Stage 6: Select 500 superior individuals and plant as advanced clonal trials (ACT)
        ACT1 = select_ind(HPT3, n_ind=n_clones_act, use="pheno", sim_param=SP)
    
    if cohort < 11:
        # Stage 7: Record ACT yields
        ACT2 = ACT1
    
    if cohort < 10:
        # Stage 8: Record ACT yields
        ACT3 = ACT2
    
    if cohort < 9:
        # Stage 9: Record ACT yields
        ACT4 = ACT3
    
    if cohort < 8:
        # Stage 10: Record ACT yields
        ACT5 = set_pheno(ACT4, var_e=var_e, reps=rep_act, sim_param=SP)
    
    if cohort < 7:
        # Stage 11: Select 40 superior individuals and plant as elite clonal trials (ECT)
        ECT1 = select_ind(ACT5, n_ind=n_clones_ect, use="pheno", sim_param=SP)
    
    if cohort < 6:
        # Stage 12: Record ECT yields
        ECT2 = ECT1
    
    if cohort < 5:
        # Stage 13: Record ECT yields
        ECT3 = ECT2
    
    if cohort < 4:
        # Stage 14: Record ECT yields
        ECT4 = ECT3
    
    if cohort < 3:
        # Stage 15: Record ECT yields
        ECT5 = ECT4
    
    if cohort < 2:
        # Stage 16: Record ECT yields
        ECT6 = set_pheno(ECT5, var_e=var_e, reps=rep_ect, p=P[cohort+13], sim_param=SP)

print("\nPipeline filled successfully!")

## Main Simulation Loop

Run the breeding program simulation with burn-in and future phases.

In [ ]:
# Create list to store results from reps
results = []

for REP in range(1, n_reps + 1):
    print(f"Working on REP: {REP}")
    
    # Create a data frame to track key parameters
    output = {
        'year': list(range(1, n_cycles + 1)),
        'rep': [REP] * n_cycles,
        'scenario': [scenario_name] * n_cycles,
        'mean_g': [0.0] * n_cycles,
        'var_g': [0.0] * n_cycles,
        'accSel': [0.0] * n_cycles
    }
    
    # Simulate year effects
    P = np.random.uniform(size=n_cycles)
    
    # Burn-in phase
    for year in range(1, n_burnin + 1):
        print(f"  Working on burnin year: {year}")
        
        # Update parents (pick new parents)
        Parents = select_ind(ECT6, n_ind=n_parents, use="pheno", sim_param=SP)
        
        # Advance year (advances yield trials by a year and collects records)
        # Stage 16
        ECT6 = set_pheno(ECT5, var_e=var_e, reps=rep_ect, p=P[year-1], sim_param=SP)
        
        # Stage 15
        ECT5 = ECT4
        
        # Stage 14
        ECT4 = ECT3
        
        # Stage 13
        ECT3 = ECT2
        
        # Stage 12
        ECT2 = ECT1
        
        # Stage 11
        ECT1 = select_ind(ACT5, n_ind=n_clones_ect, use="pheno", sim_param=SP)
        
        # Stage 10
        ACT5 = set_pheno(ACT4, var_e=var_e, reps=rep_act, p=P[year-1], sim_param=SP)
        
        # Stage 9
        ACT4 = ACT3
        
        # Stage 8
        ACT3 = ACT2
        
        # Stage 7
        ACT2 = ACT1
        
        # Stage 6
        # Calculate accuracy based on 2000 inds (n_crosses * n_progeny)
        if HPT3.n_ind > 0:
            acc_sel = np.corrcoef(HPT3.gv[:, 0], HPT3.pheno[:, 0])[0, 1]
            output['accSel'][year-1] = acc_sel
        ACT1 = select_ind(HPT3, n_ind=n_clones_act, use="pheno", sim_param=SP)
        
        # Stage 5
        HPT3 = set_pheno(HPT2, var_e=var_e, reps=rep_hpt, p=P[year-1], sim_param=SP)
        
        # Stage 4
        HPT2 = HPT1
        
        # Stage 3
        HPT1 = Seedlings
        
        # Stage 2
        Seedlings = set_pheno(F1, var_e=var_e, reps=rep_hpt, p=P[year-1], sim_param=SP)
        
        # Stage 1: Crossing block
        F1 = rand_cross(Parents, n_crosses=n_crosses, n_progeny=n_progeny, sim_param=SP)
        
        # Report results
        output['mean_g'][year-1] = mean_g(Seedlings)[0]
        output['var_g'][year-1] = var_g(Seedlings)[0]
    
    # Future phase
    for year in range(n_burnin + 1, n_burnin + n_future + 1):
        print(f"  Working on future year: {year}")
        
        # Update parents (pick new parents)
        Parents = select_ind(ECT6, n_ind=n_parents, use="pheno", sim_param=SP)
        
        # Advance year (advances yield trials by a year and collects records)
        # Stage 16
        ECT6 = set_pheno(ECT5, var_e=var_e, reps=rep_ect, p=P[year-1], sim_param=SP)
        
        # Stage 15
        ECT5 = ECT4
        
        # Stage 14
        ECT4 = ECT3
        
        # Stage 13
        ECT3 = ECT2
        
        # Stage 12
        ECT2 = ECT1
        
        # Stage 11
        ECT1 = select_ind(ACT5, n_ind=n_clones_ect, use="pheno", sim_param=SP)
        
        # Stage 10
        ACT5 = set_pheno(ACT4, var_e=var_e, reps=rep_act, p=P[year-1], sim_param=SP)
        
        # Stage 9
        ACT4 = ACT3
        
        # Stage 8
        ACT3 = ACT2
        
        # Stage 7
        ACT2 = ACT1
        
        # Stage 6
        # Calculate accuracy based on 2000 inds (n_crosses * n_progeny)
        if HPT3.n_ind > 0:
            acc_sel = np.corrcoef(HPT3.gv[:, 0], HPT3.pheno[:, 0])[0, 1]
            output['accSel'][year-1] = acc_sel
        ACT1 = select_ind(HPT3, n_ind=n_clones_act, use="pheno", sim_param=SP)
        
        # Stage 5
        HPT3 = set_pheno(HPT2, var_e=var_e, reps=rep_hpt, p=P[year-1], sim_param=SP)
        
        # Stage 4
        HPT2 = HPT1
        
        # Stage 3
        HPT1 = Seedlings
        
        # Stage 2
        Seedlings = set_pheno(F1, var_e=var_e, reps=rep_hpt, p=P[year-1], sim_param=SP)
        
        # Stage 1: Crossing block
        F1 = rand_cross(Parents, n_crosses=n_crosses, n_progeny=n_progeny, sim_param=SP)
        
        # Report results
        output['mean_g'][year-1] = mean_g(Seedlings)[0]
        output['var_g'][year-1] = var_g(Seedlings)[0]
    
    # Save results from current replicate
    results.append(output)

print("\nSimulation completed!")

## Analyze Results

Visualize the results from the simulation.

In [ ]:
# Combine results from all replicates
import pandas as pd

df = pd.DataFrame(results[0])  # For single replicate, convert dict to DataFrame

# If multiple replicates, combine them
if len(results) > 1:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

print("Results summary:")
print(df.head(10))
print(f"\nTotal years simulated: {len(df)}")

In [ ]:
# Plotting function
def plot_results(x, y, title, xlabel, ylabel, ylim=None):
    plt.plot(x, y, 'b-', linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(ylim)
    plt.grid(True, linestyle='--', alpha=0.7)

# Create plots
fig, axes = plt.subplots(3, 1, figsize=(6, 12))

# Genetic Gain
plt.sca(axes[0])
plot_results(df['year'], df['mean_g'], 
              'Genetic gain', 'Year', 'Yield')

# Genetic Variance
plt.sca(axes[1])
plot_results(df['year'], df['var_g'], 
              'Genetic variance', 'Year', 'Variance')

# Selection Accuracy
plt.sca(axes[2])
plot_results(df['year'], df['accSel'], 
              'Selection accuracy', 'Year', 'Correlation')

plt.tight_layout()
plt.savefig('PhenotypicSelection_Results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Results plot saved as 'PhenotypicSelection_Results.png'")

## Summary

This tutorial demonstrated:

1. **Founder Population Creation**: Using `runMacs2` to generate initial haplotypes
2. **Trait Definition**: Adding traits with additive, dominance, and GxE effects using `addTraitADG`
3. **Breeding Pipeline**: Setting up a 16-stage clonal breeding pipeline with:
   - Seedling evaluation (HPT stages)
   - Advanced clonal trials (ACT stages)
   - Elite clonal trials (ECT stages)
4. **Phenotypic Selection**: Selecting superior clones at each stage based on phenotypic performance
5. **Year Effects**: Incorporating year-to-year environmental variation
6. **Genetic Progress**: Tracking genetic gain, variance, and selection accuracy over time

The simulation shows how phenotypic selection can be used in clonal breeding programs to improve genetic gain over multiple generations.